In [0]:
####################################################################################
#catalog_name = "dbpraxis"
#schema_name = "create_job_artist"
#volume_name = "myfiles"
####################################################################################


In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

In [0]:
# Set the catalog and schema
spark.sql(f'USE CATALOG {catalog_name}')
spark.sql(f'USE SCHEMA {schema_name}')

In [0]:
%sql
show tables;

In [0]:
spark.sql(f'SELECT * FROM track limit 5').display()

In [0]:
%sql
--DROP TABLE IF EXISTS track_gold;
CREATE OR REPLACE TABLE track_gold AS
SELECT 
    t.id,
    t.title, 
    t.count, 
    t.rating, 
    t.len, 
    album.id as album_id, 
    artist.id as artist_id
FROM track as t 
JOIN album ON t.album = album.title
JOIN artist ON t.artist= artist.name;
select * from track_gold limit 10;

In [0]:
spark.sql("""  
WITH track_gold AS  
(SELECT
  ar.id AS artist_id,
  al.id AS album_id
FROM track_gold AS t
JOIN album  AS al ON t.album_id = al.id
JOIN artist AS ar ON t.artist_id = ar.id)
SELECT * FROM track_gold;
""").show(5)

In [0]:
spark.sql("""  
WITH track_gold AS  
(SELECT
  t.id AS track_id,
  ar.id AS artist_id
FROM track_gold AS t
JOIN album  AS al ON t.album_id = al.id
JOIN artist AS ar ON t.artist_id = ar.id)
INSERT INTO tracktoartist (track_id, artist_id) 
SELECT * FROM track_gold;
--SELECT * FROM track_gold;
""").display(5)

In [0]:
spark.sql("""    
SELECT
  t.id,
  t.title,
  ar.name AS artist,
  al.title AS album,
  t.count,
  t.rating,
  t.len
FROM track_gold AS t
JOIN album  AS al ON t.album_id = al.id
JOIN artist AS ar ON t.artist_id = ar.id
""").show(5)

In [0]:
%sql
SELECT table_schema, count(table_name)
    FROM system.information_schema.tables
    WHERE table_schema = 'tpch'
    GROUP BY table_schema
    ORDER BY 2 DESC